# R6 — Bundle: validate → deploy + run dev → deploy + run prod

Clean terminal transcript for the `claims_lakehouse` Declarative Automation Bundle at the
repo root. Order matches the acceptance requirement: **validity check**, then **deploy + run dev**
(claims_dev), then **deploy + run prod** (claims_prod). The bundle job triggers its pipeline
task automatically — the pipeline is never run manually.

| Target | Pipeline | Job | Publishes to |
| --- | --- | --- | --- |
| dev | claims-etl-pipeline-dev | claims-etl-job-dev | claims_dev.silver / claims_dev.gold |
| prod | claims-etl-pipeline-prod | claims-etl-job-prod | claims_prod.silver / claims_prod.gold |

Raw/bronze always stays in the shared source catalog `claims` — never duplicated.

In [0]:
%sh
set -e
cd /Workspace/Users/rinkimehrasalesforce@gmail.com/DE-Paper-1
/Workspace/Users/rinkimehrasalesforce@gmail.com/.local/bin/databricks bundle validate -t dev --profile paper1
/Workspace/Users/rinkimehrasalesforce@gmail.com/.local/bin/databricks bundle validate -t prod --profile paper1

In [0]:
%sh
set -e
cd /Workspace/Users/rinkimehrasalesforce@gmail.com/DE-Paper-1
/Workspace/Users/rinkimehrasalesforce@gmail.com/.local/bin/databricks bundle deploy -t dev --profile paper1
/Workspace/Users/rinkimehrasalesforce@gmail.com/.local/bin/databricks bundle run claims_etl_job -t dev --profile paper1

In [0]:
%sh
set -e
cd /Workspace/Users/rinkimehrasalesforce@gmail.com/DE-Paper-1
/Workspace/Users/rinkimehrasalesforce@gmail.com/.local/bin/databricks bundle deploy -t prod --profile paper1
/Workspace/Users/rinkimehrasalesforce@gmail.com/.local/bin/databricks bundle run claims_etl_job -t prod --profile paper1

In [0]:
# Gate: both target golds exist, are non-empty, and match the main catalog's facts
from pyspark.sql import functions as F

main_e = spark.table("claims.gold.claims_enriched")
for tgt in ["claims_dev", "claims_prod"]:
    e = spark.table(f"{tgt}.gold.claims_enriched")
    m = spark.table(f"{tgt}.gold.hospital_monthly")
    main_m = spark.table("claims.gold.hospital_monthly")
    row_delta = e.count() - main_e.count()
    amt_delta = e.agg(F.sum("billed_amount")).first()[0] - main_e.agg(F.sum("billed_amount")).first()[0]
    print(f"{tgt}.gold.claims_enriched rows={e.count()}  billed_sum_delta_vs_main={amt_delta}  row_delta={row_delta}")
    print(f"{tgt}.gold.hospital_monthly rows={m.count()}  shape={len(m.columns)} cols  (main shape={len(main_m.columns)} cols)")
    e.printSchema()
    print()

display(spark.sql("""
  SELECT 'claims' AS catalog, count(*) AS claims_enriched_rows FROM claims.gold.claims_enriched
  UNION ALL SELECT 'claims_dev', count(*) FROM claims_dev.gold.claims_enriched
  UNION ALL SELECT 'claims_prod', count(*) FROM claims_prod.gold.claims_enriched
"""))